In [1]:
import pandas as pd
df = pd.read_csv("train_ver2.csv")
df.head()

C:\Users\fgd09\AppData\Local\Temp\ipykernel_23660\2883163875.py:2: DtypeWarning: Columns (5,8,11,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("train_ver2.csv")


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,...,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-01-28,1375586,N,ES,H,35,2015-01-12,0.0,6,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
1,2015-01-28,1050611,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
2,2015-01-28,1050612,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
3,2015-01-28,1050613,N,ES,H,22,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
4,2015-01-28,1050614,N,ES,V,23,2012-08-10,0.0,35,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0


In [ ]:
#Q1 도시별
df_lastmt = df.loc[df['fecha_dato'] == '2016-05-28'].loc[:,['nomprov','ncodpers','renta','age']] # 마지막달의 데이터에서 지역,고객수,총소득 가져오기

df_nomprov = df_lastmt.groupby(['nomprov']) # 도시별로 모음
df_client = df_nomprov['ncodpers'].count()# 도시별 고객 수(합)
df_earn = df_nomprov['renta'].mean() # 도시별 평균 소득
df_mid_earn = df_nomprov['renta'].median() # 도시별 소득 중앙값
df_age = df_nomprov['age'].mean() # 도시별 평균 나이
#Q1-1
df_q1 = pd.DataFrame({
    "고객 수" : df_client,
    "평균 소득" : df_earn,
    "소득 중앙값": df_mid_earn,
    "평균 나이" : df_age
})
df_q1

#Q1-2
df_top_five = df_q1[df_q1.loc[:,'고객 수'] >= 50].sort_values(by='평균 소득', ascending=False) # 고객 수가 50명 이상인 행만 뽑고 평균 소득을 기준으로 정렬
df_top_five.head(5)

print(list(df_top_five.index)[:5])


['CEUTA', 'MADRID', 'BALEARS, ILLES', 'BARCELONA', 'MELILLA']


In [ ]:
# Q2
df_cli = df.sort_values(['ncodpers', 'fecha_dato']) #고객 코드와 날짜를 정렬
df_q2 = df_cli.iloc[:, [0, 1] + list(range(24, 48))] #iloc로 고객코드,날짜, 상품에 해당하는 열 불러옴

# 반복문에서 사용할 데이터를 리스트로 변환
clients = list(df_q2['ncodpers'])
dates = list(df_q2['fecha_dato'])

products = df.columns[24:48]
monthly_counts = {}

for c in products:
    values = list(df_q2[c])

    for i in range(1, len(values)):
        # 같은 고객인지 확인
        if clients[i] == clients[i-1]:
            # 이전에는 없고 이번에 새로 가입했는지 확인
            if values[i-1] == 0 and values[i] == 1:
                key = (dates[i], c) 

                if key not in monthly_counts:
                    monthly_counts[key] = 0

                monthly_counts[key] += 1

# 상품별 전체 신규 가입 횟수
product_counts = {}

for key in monthly_counts:
    product = key[1]

    if product not in product_counts:
        product_counts[product] = 0

    product_counts[product] += monthly_counts[key]

# 상위 5개 상품 출력
top5 = sorted(product_counts.items(), key=lambda x: x[1], reverse=True)[:5]

print(top5)



[('ind_recibo_ult1', 153205), ('ind_nom_pens_ult1', 84767), ('ind_nomina_ult1', 73799), ('ind_cco_fin_ult1', 69997), ('ind_tjcr_fin_ult1', 69311)]


In [ ]:
def chi_square(df): #카이제곱 검정 함수
    col_total = df.sum() #df의 열별 합계 계산 -> total로 전체 몇명인지 계산
    total = col_total.sum()
    df_expected = df.copy()

    for row in df.index: # 기댓값 계산 - 각 행과 열을 불러와서 해당하는 기댓값 계산
        for col in df.columns:
            df_expected.loc[row, col] = (df.loc[row].sum() * col_total[col] / total)

    df_chi = (df - df_expected) ** 2 / df_expected # 카이제곱 계산

    chi2 = df_chi.sum().sum() #해당하는 카이제곱 열별 계산 후 모든 카이제곱 합치기

    return chi2

In [ ]:
#Q3
products = list(range(24,48)) # 상품에 해당하는 인덱스 번호 리스트로 생성
df_last = df.loc[df['fecha_dato'] == '2016-05-28'].iloc[:,products] # iloc를 이용해서 마지막 달에 해당하는 상품 정보 df로 따로 불러옴
df_last_info = df.loc[df['fecha_dato'] == '2016-05-28'].loc[:,['sexo','age','segmento','ind_empleado','tiprel_1mes']] # 마지막 달에 해당하는 고객 정보를 df로 불러옴
df_last['가입 상품수'] = df_last.sum(axis=1) # 불러온 상품 정보를 행별 계산(axis=1)을 통해서 가입 상품 수를 계산
df_last_info['3개 이상'] = df_last['가입 상품수'].apply(lambda x: 1 if x >=3 else 0) #apply를 이용해서 가입 상품수가 3개 이상이면 1(True), 3개 미만이면 0(False)로 

#Q3 - 성별 

df_sexo_total = df_last_info.loc[:, ['sexo', '3개 이상']].groupby('sexo')['3개 이상'].count() # 성별정보와 3개이상 정보를 불러와서 groupby로 성별별로 묶고 count를 통해서 성별별로 총 몇명인지 계산

df_sexo = df_last_info.loc[:, ['sexo', '3개 이상']].groupby('sexo').sum() # 3개이상인 부분은 1이므로 sum을 통해서 3개이상이 몇명인지 성별별로 계산

df_sexo['3개 미만'] = df_sexo_total - df_sexo['3개 이상'] # 3개미만 계산

chi2_sexo = chi_square(df_sexo.loc[:, ['3개 미만', '3개 이상']]) #카이제곱 계산
chi2_sexo


C:\Users\fgd09\AppData\Local\Temp\ipykernel_23660\2272450940.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '369619.74558107374' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_expected.loc[row, col] = (
C:\Users\fgd09\AppData\Local\Temp\ipykernel_23660\2272450940.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '56246.25441892623' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_expected.loc[row, col] = (


np.float64(3412.5902283331347)

In [ ]:
def to_age(age): #age가 숫자로 표현되어있으니 문자로 바꿔주는 함수
    if age >= 65:
        return 'senior'
    elif 30 <= age < 65:
        return 'adult'
    elif age < 30:
        return 'young'

df_last_info['나이'] = df_last_info['age'].apply(to_age) #apply를 이용해서 df에 나이를 다 문자로 바꿔줌

# 나이 그룹별 전체 고객 수
df_age_total = df_last_info.loc[:, ['나이', '3개 이상']].groupby('나이')['3개 이상'].count()

# 나이 그룹별 3개 이상인 고객 수
df_age = df_last_info.loc[:, ['나이', '3개 이상']].groupby('나이').sum()

# 나이 그룹별 3개 미만인 고객 수
df_age['3개 미만'] = df_age_total - df_age['3개 이상']

# 관측 빈도표
df_age = df_age.loc[:, ['3개 미만', '3개 이상']]

df_age

chi2_age = chi_square(df_age)

print('나이 그룹 카이제곱 통계량:', chi2_age)

나이 그룹 카이제곱 통계량: 61002.2024141415


C:\Users\fgd09\AppData\Local\Temp\ipykernel_23660\2272450940.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '426064.72768889036' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_expected.loc[row, col] = (
C:\Users\fgd09\AppData\Local\Temp\ipykernel_23660\2272450940.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '64835.272311109635' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_expected.loc[row, col] = (


In [ ]:
# segmento별 전체 고객 수
df_segment_total = df_last_info.loc[:, ['segmento', '3개 이상']].groupby('segmento')['3개 이상'].count()

# segmento별 3개 이상인 고객 수
df_segment = df_last_info.loc[:, ['segmento', '3개 이상']].groupby('segmento').sum()

# segmento별 3개 미만인 고객 수
df_segment['3개 미만'] = df_segment_total - df_segment['3개 이상']

df_segment = df_segment.loc[:, ['3개 미만', '3개 이상']]

chi2_segment = chi_square(df_segment)

print('segmento 카이제곱 통계량:', chi2_segment)

segmento 카이제곱 통계량: 124374.54879922631


C:\Users\fgd09\AppData\Local\Temp\ipykernel_23660\2272450940.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '30703.231371183047' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_expected.loc[row, col] = (
C:\Users\fgd09\AppData\Local\Temp\ipykernel_23660\2272450940.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4711.768628816955' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_expected.loc[row, col] = (


In [ ]:
# Q3 - ind_empleado

# 직원 구분별 전체 고객 수
df_empleado_total = df_last_info.loc[:, ['ind_empleado', '3개 이상']].groupby('ind_empleado')['3개 이상'].count()

# 직원 구분별 3개 이상인 고객 수
df_empleado = df_last_info.loc[:, ['ind_empleado', '3개 이상']].groupby('ind_empleado').sum()

# 직원 구분별 3개 미만인 고객 수
df_empleado['3개 미만'] = df_empleado_total - df_empleado['3개 이상']

df_empleado = df_empleado.loc[:, ['3개 미만', '3개 이상']]

df_empleado

chi2_empleado = chi_square(df_empleado)

print('ind_empleado 카이제곱 통계량:', chi2_empleado)

ind_empleado 카이제곱 통계량: 1343.689088455891


C:\Users\fgd09\AppData\Local\Temp\ipykernel_23660\2272450940.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '128.45300406998527' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_expected.loc[row, col] = (
C:\Users\fgd09\AppData\Local\Temp\ipykernel_23660\2272450940.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '19.54699593001472' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_expected.loc[row, col] = (


In [10]:
# Q3 - tiprel_1mes

# 고객 관계 유형별 전체 고객 수
df_triple_total = df_last_info.loc[:, ['tiprel_1mes', '3개 이상']].groupby('tiprel_1mes')['3개 이상'].count()

# 고객 관계 유형별 3개 이상인 고객 수
df_triple = df_last_info.loc[:, ['tiprel_1mes', '3개 이상']].groupby('tiprel_1mes').sum()

# 고객 관계 유형별 3개 미만인 고객 수
df_triple['3개 미만'] = df_triple_total - df_triple['3개 이상']

# 관측 빈도표
df_triple = df_triple.loc[:, ['3개 미만', '3개 이상']]

df_triple

chi2_triple = chi_square(df_triple)

print('tiprel_1mes 카이제곱 통계량:', chi2_triple)

tiprel_1mes 카이제곱 통계량: 183808.5915229053


C:\Users\fgd09\AppData\Local\Temp\ipykernel_23660\2272450940.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '340540.51401110366' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_expected.loc[row, col] = (
C:\Users\fgd09\AppData\Local\Temp\ipykernel_23660\2272450940.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '52066.485988896355' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_expected.loc[row, col] = (


In [ ]:
#Q3
chi_results = pd.Series({
    'sexo': chi2_sexo,
    'age': chi2_age,
    'segmento': chi2_segment,
    'ind_empleado': chi2_empleado,
    'tiprel_1mes': chi2_triple
})

chi_results.sort_values(ascending=False)


tiprel_1mes     183808.591523
segmento        124374.548799
age              61002.202414
sexo              3412.590228
ind_empleado      1343.689088
dtype: float64